# Design principles in Python

Design principles matter more as code grows. At the beginner level, the main challenge is getting code to run. At the intermediate level, the challenge becomes keeping code understandable, extensible, and easy to test.

This module takes common design ideas—such as separating responsibilities, depending on abstractions instead of concrete implementations, and designing plugin-friendly systems—and translates them into Python-shaped code.

While reading these notebooks, keep asking what would make a design easier to change later. Good design is usually less about cleverness and more about reducing unnecessary dependencies between parts.

## Visual model

```text
small interfaces -> loose coupling -> easier testing and extension
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. SOLID, translated

### S — Single responsibility

*A module should have one reason to change.* Survives intact, and applies to
functions and modules at least as much as to classes.

The practical test is not "does this class do one thing" (unanswerably vague) but
**"who asks for changes to this?"** If the finance team and the marketing team
both file tickets against one class, it has two responsibilities.

In [ ]:
class Report:                     # three reasons to change
    def fetch(self): ...          # the database team
    def calculate(self): ...      # the finance team
    def render_pdf(self): ...     # the design team

### O — Open/closed

*Open for extension, closed for modification.* In Java this means inheritance and
interfaces. **In Python it usually means a function argument.**

In [ ]:
# closed for modification: you never edit this
def process(items, transform=lambda x: x, key=None, on_error=None): ...

# extension is a registry, not a subclass
HANDLERS: dict[str, Callable[[Event], None]] = {}

def handles(event_type: str):
    def register(fn):
        HANDLERS[event_type] = fn
        return fn
    return register

@handles("click")
def on_click(event): ...

That registry is the Strategy pattern, the Command pattern, and half of the
Visitor pattern, in eight lines and with no classes.

### L — Liskov substitution

Covered in Module 10, and it applies unchanged. Python's dynamism makes it easier
to violate and no less costly when you do — the failure just moves from compile
time to runtime.

### I — Interface segregation

*No client should depend on methods it does not use.* This is where `Protocol`
shines (Module 10): define the **narrowest** interface your function actually
needs.

In [ ]:
class Readable(Protocol):
    def read(self, n: int) -> bytes: ...

def parse(source: Readable) -> Document: ...   # not "a File", just "readable"

Now a file, a socket, a `BytesIO`, and a test fake all qualify. Typing the
parameter as a concrete class instead would have excluded three of them for no
reason.

### D — Dependency inversion

*Depend on abstractions, not concretions.* In Python, the abstraction is usually
a **function parameter**, not an interface hierarchy:

In [ ]:
# concrete dependency: untestable without a database and a clock
class OrderService:
    def __init__(self):
        self.db = PostgresConnection("prod")
        self.clock = datetime.now

# inverted: the caller supplies both
class OrderService:
    def __init__(self, db: SupportsQuery, now: Callable[[], datetime] = datetime.now):
        self.db = db
        self.now = now

That second version is testable with a dict and a lambda. No framework, no
container, no annotations — this **is** dependency injection, and in Python it
needs no library.

---

## Concept 2. Patterns that Python dissolves

| Pattern | In Java | In Python |
|---|---|---|
| **Strategy** | An interface + N classes | Pass a function |
| **Command** | An interface + N classes | Pass a function, or `partial` |
| **Factory** | A factory class | A function, or a `@classmethod` |
| **Abstract Factory** | Two class hierarchies | A dict of constructors |
| **Singleton** | Private ctor + static instance | A module. Modules are singletons. |
| **Decorator** | Wrapper class hierarchy | `@decorator` (Module 15) |
| **Observer** | Listener interfaces | A list of callables |
| **Iterator** | An interface | `__iter__` / `yield` (Module 14) |
| **Template Method** | Abstract base + hooks | A function taking hook functions |
| **Adapter** | A wrapper class | Often just duck typing |
| **Builder** | A builder class | Keyword arguments, or a frozen dataclass + `replace` |
| **Visitor** | Double dispatch | `match`, or `functools.singledispatch` |

The patterns that remain useful in Python — because they solve a real structural
problem rather than a missing language feature — are Adapter (when the interfaces
genuinely differ), Facade, Proxy, Repository, and Unit of Work.

**Singleton deserves a note**, because it is the one people reach for most and
need least:

In [ ]:
# config.py
_settings = load_settings()

def get_settings() -> Settings:
    return _settings

A module is imported once per process and cached in `sys.modules` (Module 06).
That is a singleton, with none of the thread-safety problems of the
double-checked-locking version and none of the testability problems of a class
that hides its own construction.

---

## Concept 4. Descriptors

The mechanism under `@property`, `@classmethod`, `@staticmethod`,
`cached_property`, and every ORM field you have ever used.

In [ ]:
class Positive:
    """A reusable validated attribute."""

    def __set_name__(self, owner: type, name: str) -> None:
        self._name = f"_{name}"          # called at CLASS creation time

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self                   # accessed on the CLASS, not an instance
        return getattr(obj, self._name)

    def __set__(self, obj, value) -> None:
        if value <= 0:
            raise ValueError(f"{self._name[1:]} must be positive, got {value}")
        setattr(obj, self._name, value)


class Product:
    price = Positive()          # written ONCE
    weight = Positive()
    quantity = Positive()

Three properties would have been thirty lines of near-identical code. The
descriptor is written once and reused.

**Data versus non-data descriptors** (Module 08, exercise 1): defining both
`__get__` and `__set__` makes it a *data* descriptor, which takes priority over
the instance `__dict__`. Defining only `__get__` makes it *non-data*, which the
instance dict beats — and that asymmetry is precisely how `cached_property`
works.

**When to use a descriptor:** the same attribute logic repeated across three or
more attributes, or across several classes. Below that, `@property` is clearer.

---

## Concept 7. When not to use a class at all

The most common over-engineering in Python is a class that should be a function.

In [ ]:
class EmailValidator:                    # a class with no state
    def __init__(self, strict=False):
        self.strict = strict
    def validate(self, email): ...

def validate_email(email: str, *, strict: bool = False) -> bool: ...

**Signs you wanted a function:**

- The class has one public method, and it is not `__call__`.
- `__init__` only stores arguments the single method uses.
- It is constructed and used on the same line.
- Every method is a `@staticmethod`.
- Its name ends in `-er` or `-Manager` and describes an action rather than a
  thing.

**A class earns its place when** it holds state across calls, has several
operations over shared data, participates in a protocol (Module 09), needs
several implementations behind one interface, or manages a resource lifecycle.

The corollary: a module full of functions is a perfectly good design. Python is
not Java; there is no requirement that everything live inside a class.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: SOLID, translated
- Section 2: Patterns that Python dissolves
- Section 3: Composition over inheritance, concretely
- Section 4: Descriptors
- Section 5: `__init_subclass__` and class decorators
- Section 6: Metaclasses
- Section 7: When not to use a class at all

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import threading
from abc import ABC, abstractmethod
from typing import Any


# --- 1: Strategy --------------------------------------------------------------

---

## `SortStrategy`

_SortStrategy_

In [ ]:
class SortStrategy(ABC):
    @abstractmethod
    def sort(self, data: list[int]) -> list[int]: ...

---

## `AscendingSort`

_AscendingSort_

In [ ]:
class AscendingSort(SortStrategy):
    def sort(self, data: list[int]) -> list[int]:
        return sorted(data)

---

## `DescendingSort`

_DescendingSort_

In [ ]:
class DescendingSort(SortStrategy):
    def sort(self, data: list[int]) -> list[int]:
        return sorted(data, reverse=True)

---

## `ByAbsoluteValueSort`

_ByAbsoluteValueSort_

In [ ]:
class ByAbsoluteValueSort(SortStrategy):
    def sort(self, data: list[int]) -> list[int]:
        return sorted(data, key=abs)

---

## `Sorter`

_Sorter_

In [ ]:
class Sorter:
    def __init__(self, strategy: SortStrategy) -> None:
        self._strategy = strategy

    def sort(self, data: list[int]) -> list[int]:
        return self._strategy.sort(data)

---

## `ConfigSingleton`

_ConfigSingleton_

In [ ]:
class ConfigSingleton:
    _instance: ConfigSingleton | None = None
    _lock = threading.Lock()

    def __new__(cls) -> ConfigSingleton:
        if cls._instance is None:
            with cls._lock:
                if cls._instance is None:      # double-checked locking
                    cls._instance = super().__new__(cls)
                    cls._instance._data = {"debug": False}
        return cls._instance

    def get(self, key: str) -> Any:
        return self._data.get(key)             # type: ignore[attr-defined]

---

## `Animal`

_Animal_

In [ ]:
class Animal(ABC):
    @abstractmethod
    def speak(self) -> str: ...

---

## `Dog`

_Dog_

In [ ]:
class Dog(Animal):
    def speak(self) -> str: return "woof"

---

## `Cat`

_Cat_

In [ ]:
class Cat(Animal):
    def speak(self) -> str: return "meow"

---

## `AnimalFactory`

_AnimalFactory_

In [ ]:
class AnimalFactory:
    @staticmethod
    def create(kind: str) -> Animal:
        if kind == "dog":
            return Dog()
        if kind == "cat":
            return Cat()
        raise ValueError(f"unknown animal: {kind}")

---

## `Observer`

_Observer_

In [ ]:
class Observer(ABC):
    @abstractmethod
    def update(self, event: str) -> None: ...

---

## `EmailObserver`

_EmailObserver_

In [ ]:
class EmailObserver(Observer):
    def update(self, event: str) -> None:
        print(f"    email: {event}")

---

## `LogObserver`

_LogObserver_

In [ ]:
class LogObserver(Observer):
    def update(self, event: str) -> None:
        print(f"    log: {event}")

---

## `Subject`

_Subject_

In [ ]:
class Subject:
    def __init__(self) -> None:
        self._observers: list[Observer] = []

    def attach(self, observer: Observer) -> None:
        self._observers.append(observer)

    def detach(self, observer: Observer) -> None:
        self._observers.remove(observer)

    def notify(self, event: str) -> None:
        for observer in self._observers:
            observer.update(event)

---

## `DataProcessor`

_DataProcessor_

In [ ]:
class DataProcessor(ABC):
    def process(self, raw: str) -> str:
        data = self.parse(raw)
        data = self.transform(data)
        return self.render(data)

    @abstractmethod
    def parse(self, raw: str) -> list[str]: ...

    def transform(self, data: list[str]) -> list[str]:
        return data

    @abstractmethod
    def render(self, data: list[str]) -> str: ...

---

## `CsvProcessor`

_CsvProcessor_

In [ ]:
class CsvProcessor(DataProcessor):
    def parse(self, raw: str) -> list[str]:
        return raw.split(",")

    def transform(self, data: list[str]) -> list[str]:
        return [d.strip().upper() for d in data]

    def render(self, data: list[str]) -> str:
        return " | ".join(data)

---

## `Pizza`

_Pizza_

In [ ]:
class Pizza:
    def __init__(self) -> None:
        self.size = ""
        self.toppings: list[str] = []
        self.extra_cheese = False

---

## `PizzaBuilder`

_PizzaBuilder_

In [ ]:
class PizzaBuilder:
    def __init__(self) -> None:
        self._pizza = Pizza()

    def size(self, size: str) -> PizzaBuilder:
        self._pizza.size = size
        return self

    def topping(self, topping: str) -> PizzaBuilder:
        self._pizza.toppings.append(topping)
        return self

    def extra_cheese(self) -> PizzaBuilder:
        self._pizza.extra_cheese = True
        return self

    def build(self) -> Pizza:
        if not self._pizza.size:
            raise ValueError("size is required")
        return self._pizza

---

## `demo`

_demo_

In [ ]:
def demo() -> None:
    print("1 Strategy:  ", Sorter(ByAbsoluteValueSort()).sort([-5, 2, -1]))
    print("2 Singleton: ", ConfigSingleton() is ConfigSingleton())
    print("3 Factory:   ", AnimalFactory.create("dog").speak())
    s = Subject(); s.attach(EmailObserver()); s.attach(LogObserver())
    print("4 Observer:"); s.notify("order placed")
    print("5 Template:  ", CsvProcessor().process("a, b, c"))
    p = PizzaBuilder().size("large").topping("olive").extra_cheese().build()
    print("6 Builder:   ", p.size, p.toppings, p.extra_cheese)

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    demo()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.